# 01 · Lectura e integración de Work-Olist

De archivos tabulares a datos comprensibles: rutas, tipos, unidades de observación, filtros, agregaciones y uniones relacionales seguras.

**Modalidad:** práctica guiada, código ejecutable, interpretación y ejercicios. **Datos:** nueve CSV aportados por el usuario en `data/raw/`, con el esquema relacional de Olist. Se calculan resultados sobre estos archivos, sin simular registros ni cambiar las fuentes.

## Objetivos y prerrequisitos
Al finalizar podrás: (i) distinguir archivo, tabla, fila, columna y variable; (ii) importar CSV y diagnosticar tipos; (iii) reconocer llaves y cardinalidades; (iv) filtrar, ordenar, agrupar y unir sin duplicar pedidos; (v) exportar un resultado con trazabilidad. Se explican `Series`, `DataFrame`, `list`, `dict`, `loc`, `groupby` y `merge` sin suponer experiencia previa.

**Procedencia y acceso.** Los nueve CSV se entregaron en este proyecto en `data/raw/`. Su esquema es el publicado para el [conjunto Brazilian E-Commerce de Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce). El análisis utiliza los archivos suministrados, no presupone una descarga ni verifica su identidad byte a byte con Kaggle. El proyecto conserva los nombres de los CSV recibidos y unifica las operaciones en `src/olist_tools.py`.

**Pregunta guía:** ¿cuántos pedidos, compradores y líneas de producto hay, y qué pasa si confundimos sus unidades de análisis?

In [1]:
from pathlib import Path
import sys
from IPython.display import display
import numpy as np
import pandas as pd

# Estructura real del proyecto:
# DMEMCD_2026/Python/*.ipynb y Python/olist_tools.py
# DMEMCD_2026/work-at-olist-data/datasets/*.csv
# Funciona si el kernel arranca en la raíz o en alguna subcarpeta.
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "Python" / "olist_tools.py").is_file()
             and (p / "work-at-olist-data" / "datasets").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No encuentro Python/olist_tools.py y work-at-olist-data/datasets/. "
        "Abre en VS Code la carpeta DMEMCD_2026 y selecciona su kernel."
    )
PYTHON_DIR = ROOT / "Python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))
from olist_tools import (read_tables, build_order_level, build_item_level,
                        table_inventory, missing_profile, safe_minmax, safe_zscore)
TABLAS, ORIGEN = read_tables(ROOT)
print("ORIGEN:", ORIGEN)
print("Directorio raíz:", ROOT)


ORIGEN: archivos CSV suministrados por el usuario (esquema Olist)
Directorio de trabajo: /mnt/data/Work_Olist_Notebooks_2026


### Diccionario relacional y unidades de observación

| Tabla | Unidad de observación | Llave o criterio de identidad | Precaución principal |
|---|---|---|---|
| `orders` | un pedido | `order_id` | el estado condiciona la disponibilidad de fechas |
| `customers` | registro de cliente vinculado al pedido | `customer_id` | no equivale a `customer_unique_id` |
| `items` | línea de pedido | (`order_id`, `order_item_id`) | varios ítems por pedido |
| `payments` | registro de pago | (`order_id`, `payment_sequential`) | varios pagos por pedido |
| `reviews` | reseña asociada a pedido | (`review_id`, `order_id`) | puede haber varias reseñas por pedido |
| `products` | producto | `product_id` | categorías y atributos físicos pueden faltar |
| `sellers` | vendedor | `seller_id` | estado es etiqueta geográfica |
| `translation` | categoría traducida | `product_category_name` | no asumir cobertura de todas las categorías |
| `geolocation` | coordenada registrada para prefijo postal | no hay llave única de fila | un prefijo postal puede tener muchos puntos |

**Advertencia sobre unidades:** sumar pagos sobre una tabla expandida por ítems duplica importes. Agregar primero las tablas de detalle por `order_id` preserva una fila por pedido.


## 1. Reconocer los archivos antes de leer los datos
Una ruta relativa permite trabajar en distintas computadoras. En Python, `Path` representa rutas; `pandas.read_csv` lee tablas y conserva los identificadores como texto para no confundirlos con cantidades. Un CSV no guarda información de tipos tan rica como una base relacional: debemos diagnosticar después de importar.

In [2]:
# Datos reales: conteos y llaves comprobables en los archivos aportados.
resumen_fuente = pd.DataFrame([
    {"tabla": nombre, "registros": len(df), "columnas": df.shape[1]}
    for nombre, df in TABLAS.items()
]).sort_values("tabla")
display(resumen_fuente)
assert TABLAS["orders"].order_id.is_unique
assert TABLAS["customers"].customer_id.is_unique
print("Compradores distintos (ID anonimizado):", TABLAS["customers"].customer_unique_id.nunique())

,tabla,registros,columnas
1,customers,99441,5
2,items,112650,7
0,orders,99441,8
3,payments,103886,5
5,products,32951,9
4,reviews,99224,7
6,sellers,3095,4
7,translation,71,2


Compradores distintos (ID anonimizado): 96096


In [3]:
print("Archivos esperados (solo nombres):")
for nombre, tabla in TABLAS.items():
    print(f"  {nombre:12s}: {tabla.shape[0]:>7,} filas × {tabla.shape[1]} columnas")
print("\nInventario y memoria aproximada:")
display(table_inventory(TABLAS))

Archivos esperados (solo nombres):
  orders      :  99,441 filas × 8 columnas
  customers   :  99,441 filas × 5 columnas
  items       : 112,650 filas × 7 columnas
  payments    : 103,886 filas × 5 columnas
  reviews     :  99,224 filas × 7 columnas
  products    :  32,951 filas × 9 columnas
  sellers     :   3,095 filas × 4 columnas
  translation :      71 filas × 2 columnas

Inventario y memoria aproximada:


,tabla,filas,columnas,nulos_en_celdas,memoria_MiB
0,customers,99441,5,0,30.93
1,items,112650,7,0,34.06
2,orders,99441,8,4908,24.66
3,payments,103886,5,0,16.23
4,products,32951,9,2448,6.30
5,reviews,99224,7,145903,27.77
6,sellers,3095,4,0,0.72
7,translation,71,2,0,0.01


### 1.1. Novena tabla: geolocalización sin sobrecargar memoria

Las ocho primeras tablas pueden integrarse por claves comerciales. La novena, `geolocation`, tiene **1 000 163 registros** y múltiples coordenadas para un mismo prefijo postal. En esta primera lectura mostramos únicamente cinco filas y contamos líneas en el CSV, **sin unir cada punto a los pedidos**. En el notebook 3 aprenderemos a resumir coordenadas antes de visualizarlas.

Un archivo extenso no exige cargarlo entero para conocer su estructura. Si más adelante se requieren cálculos geográficos, puede usarse `read_csv(usecols=...)`, lectura por bloques (`chunksize`) o agregación por prefijo.

In [4]:
# Inspección eficiente de geolocation: solo encabezado, cinco filas y conteo físico.
archivo_geo = ROOT / "work-at-olist-data" / "datasets" / "olist_geolocation_dataset.csv"
print("Geolocalización: filas =", sum(1 for _ in archivo_geo.open(encoding="utf-8")) - 1)
display(pd.read_csv(archivo_geo, nrows=5))
print("No se realiza ninguna unión geográfica en esta etapa.")

Geolocalización: filas = 1000163


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


No se realiza ninguna unión geográfica en esta etapa.


## 2. Vistas previas, estructura y significado
`head()` devuelve las primeras filas; `info()` resume tipos y nulos; `describe()` describe magnitudes y categorías. **No confundir número almacenado con magnitud:** un prefijo postal no debe promediarse. No mostrar aquí los textos libres de reseñas.

In [5]:
pedidos = TABLAS["orders"].copy()
clientes = TABLAS["customers"].copy()
items = TABLAS["items"].copy()
pagos = TABLAS["payments"].copy()
display(pedidos.head(4))
print("\nTipos y completitud de pedidos:")
pedidos.info()
display(missing_profile(pedidos).head(9))
print("\nResumen de precios de líneas de pedido (BRL, sin flete):")
display(items["price"].describe(percentiles=[.1,.25,.5,.75,.9,.99]).to_frame())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15



Tipos y completitud de pedidos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  string        
 1   customer_id                    99441 non-null  string        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(1), string(2)
memory usage: 6.1+ MB


,variable,faltantes,porcentaje,tipo
0,order_delivered_customer_date,2965,2.98,datetime64[ns]
1,order_delivered_carrier_date,1783,1.79,datetime64[ns]
2,order_approved_at,160,0.16,datetime64[ns]
3,order_id,0,0.00,string
4,order_purchase_timestamp,0,0.00,datetime64[ns]
5,order_status,0,0.00,object
6,customer_id,0,0.00,string
7,order_estimated_delivery_date,0,0.00,datetime64[ns]



Resumen de precios de líneas de pedido (BRL, sin flete):


,price
count,112650.000000
mean,120.653739
std,183.633928
min,0.850000
10%,23.800000
25%,39.900000
50%,74.990000
75%,134.900000
90%,229.800000
99%,890.000000


### Tipos de datos y estructuras computacionales
Una **lista** preserva orden, un **diccionario** asocia claves con valores, una **tupla** es una secuencia inmutable y un `numpy.ndarray` permite cálculo vectorizado. `Series` y `DataFrame` agregan etiquetas y valores faltantes. Esta cultura computacional sirve para elegir estructuras sencillas antes de crear clases o patrones complejos.

In [6]:
columnas = ["order_id", "order_status"]       # lista
config = {"tabla":"orders", "llave":"order_id"} # diccionario
etapas = ("leer", "auditar", "unir")             # tupla
precios = items["price"].dropna().to_numpy()     # ndarray
print(type(columnas).__name__, type(config).__name__, type(etapas).__name__, type(precios).__name__)
print("Promedio vectorizado (ilustración, no limpieza):", np.mean(precios).round(2))

list dict tuple ndarray
Promedio vectorizado (ilustración, no limpieza): 120.65


## 3. Identificadores, unidades y cardinalidad
`order_id` identifica un pedido, `customer_id` identifica el registro de cliente asociado al pedido y `customer_unique_id` aproxima un comprador recurrente. En `items`, `payments` y `reviews` es legítimo encontrar varias filas por pedido. Una unión `one_to_many` multiplica filas; si luego sumamos `payment_value` desde esa tabla expandida, podemos duplicar ingresos **por error**.

In [7]:
chequeos = pd.DataFrame([
  {"tabla":"orders","llave":"order_id","filas":len(pedidos),"llaves_unicas":pedidos.order_id.nunique()},
  {"tabla":"items","llave":"order_id + order_item_id","filas":len(items),
   "llaves_unicas":items[["order_id","order_item_id"]].drop_duplicates().shape[0]},
  {"tabla":"payments","llave":"order_id + payment_sequential","filas":len(pagos),
   "llaves_unicas":pagos[["order_id","payment_sequential"]].drop_duplicates().shape[0]},
  {"tabla":"customers","llave":"customer_id","filas":len(clientes),"llaves_unicas":clientes.customer_id.nunique()},
])
display(chequeos)
print("Pedidos / registros de cliente / compradores únicos:",
      pedidos.order_id.nunique(),clientes.customer_id.nunique(),clientes.customer_unique_id.nunique())
assert pedidos.order_id.is_unique

,tabla,llave,filas,llaves_unicas
0,orders,order_id,99441,99441
1,items,order_id + order_item_id,112650,112650
2,payments,order_id + payment_sequential,103886,103886
3,customers,customer_id,99441,99441


Pedidos / registros de cliente / compradores únicos: 99441 99441 96096


## 4. Filtrar, ordenar, seleccionar y crear variables
`loc[filas, columnas]` deja explícito el criterio de selección; `.assign()` genera columnas sin sobrescribir innecesariamente los datos originales. Las condiciones se combinan con `&` (y), `|` (o) y paréntesis. Evita `for` fila por fila cuando una expresión vectorizada resuelve el problema.

In [8]:
# Un filtro por condición y columnas, sin modificar "pedidos".
sp_entregados = pedidos.loc[
    pedidos["order_status"].eq("delivered") &
    pedidos["customer_id"].isin(clientes.loc[clientes["customer_state"].eq("SP"), "customer_id"]),
    ["order_id","customer_id","order_purchase_timestamp","order_status"]
].sort_values("order_purchase_timestamp")
print("Entregados en SP:",len(sp_entregados))
display(sp_entregados.head(5))
lineas = items.assign(valor_con_flete=lambda d: d["price"] + d["freight_value"])
display(lineas[["order_id","order_item_id","price","freight_value","valor_con_flete"]].head())

Entregados en SP: 40501


,order_id,customer_id,order_purchase_timestamp,order_status
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,2016-09-15 12:16:38,delivered
93285,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,2016-10-03 09:44:50,delivered
88472,d207cc272675637bfed0062edffd0818,b8cf418e97ae795672d326288dfab7a7,2016-10-03 22:06:03,delivered
6747,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,2016-10-03 22:31:31,delivered
78848,0a0837a5eee9e7a9ce2b1fa831944d27,4f3f778022aefa22b9f9e52d2c47edf3,2016-10-04 09:06:10,delivered


,order_id,order_item_id,price,freight_value,valor_con_flete
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


## 5. Groupby, resumen y tablas de doble entrada
`groupby` separa, aplica y combina. Para comparar estados es necesario recordar que los conteos son **pedidos**, no personas; los estados no necesariamente tienen la misma población. `crosstab` es útil con variables nominales u ordinales.

In [9]:
por_estado = (pedidos.merge(clientes[["customer_id","customer_state"]],on="customer_id",validate="many_to_one")
              .groupby("customer_state").agg(pedidos=("order_id","nunique"))
              .sort_values("pedidos",ascending=False))
display(por_estado.head(10))
ct = pd.crosstab(pedidos["order_status"], columns="pedidos", normalize="columns")
display(ct.rename(columns={"pedidos":"proporcion"}).round(3))

,pedidos
customer_state,
SP,41746
RJ,12852
MG,11635
RS,5466
PR,5045
SC,3637
BA,3380
DF,2140
ES,2033


col_0,proporcion
order_status,
approved,0.000
canceled,0.006
created,0.000
delivered,0.970
invoiced,0.003
processing,0.003
shipped,0.011
unavailable,0.006


## 6. Uniones con `validate` e `indicator`
**Caso seguro:** unir pedidos con clientes mediante `many_to_one`. **Caso inseguro:** unir pedidos con ítems y pagos a la vez sin agregación previa. `indicator=True` ayuda a identificar ausencias en las llaves y `validate=` detecta relaciones no previstas. Para construir una tabla analítica de pedidos agregaremos ítems, pagos y reseñas **antes** de unir.

In [10]:
prueba_merge = pedidos[["order_id","customer_id"]].merge(
    clientes[["customer_id","customer_state"]], on="customer_id", how="left",
    validate="many_to_one", indicator=True)
display(prueba_merge["_merge"].value_counts(dropna=False).to_frame("filas"))
analitica = build_order_level(TABLAS)
print("Filas en pedidos originales:",len(pedidos),"| filas en tabla analítica:",len(analitica))
assert len(analitica)==len(pedidos)
assert analitica.order_id.is_unique
print("Cada fila sigue representando un pedido.")
display(analitica[["order_id","n_items","n_pagos","subtotal_productos","total_flete","total_pagado"]].head())

,filas
_merge,
both,99441
left_only,0
right_only,0


Filas en pedidos originales: 99441 | filas en tabla analítica: 99441
Cada fila sigue representando un pedido.


,order_id,n_items,n_pagos,subtotal_productos,total_flete,total_pagado
0,e481f51cbdc54678b7cc49136f2d6af7,1.0,3.0,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,1.0,1.0,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,1.0,1.0,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,1.0,1.0,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,1.0,1.0,19.90,8.72,28.62


### Por qué las sumas pueden no coincidir exactamente
`sum(order_items.price)` es el subtotal de productos y **no** incluye flete; `sum(payment_value)` puede incorporar flete, descuentos, reembolsos u otras diferencias operativas. La tabla original no debe forzarse a igualdad por conveniencia. Los pedidos cancelados y estados incompletos requieren interpretación separada.

In [11]:
comparacion = analitica[["subtotal_productos","total_flete","total_pagado"]].sum(min_count=1)
display(comparacion.rename("total_BRL").round(2).to_frame())
# La tabla de líneas mantiene una unidad distinta.
items_categorias = build_item_level(TABLAS)
print("Filas en ítems (antes/después de unir catálogos):",len(items),len(items_categorias))
assert len(items_categorias)==len(items)
display(items_categorias[["order_id","product_id","categoria_mostrada","price"]].head())

,total_BRL
subtotal_productos,13591643.70
total_flete,2256599.44
total_pagado,16008872.12


Filas en ítems (antes/después de unir catálogos): 112650 112650


,order_id,product_id,categoria_mostrada,price
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58.90
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239.90
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,199.00
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,perfumery,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,garden_tools,199.90


## 7. Fechas y variables derivadas
Convertir fechas con `pd.to_datetime(errors='coerce')` permite construir duración, mes y retraso. **No interpretes NA en fecha de entrega como cero días**: un pedido no entregado no tiene duración observada. La interpretación temporal depende también de convenciones horarias y eventos de negocio.

In [12]:
columnas = ["order_id","order_status","mes_compra","dias_hasta_entrega","dias_retraso","entrega_tardia"]
display(analitica[columnas].head(8))
print("Tasa tardía entre pedidos con duración observada:",
      f"{analitica['entrega_tardia'].dropna().mean():.1%}")

,order_id,order_status,mes_compra,dias_hasta_entrega,dias_retraso,entrega_tardia
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10,8.436574,-7.107488,False
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07,13.782037,-5.355729,False
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08,9.394213,-17.245498,False
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11,13.208750,-12.980069,False
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02,2.873877,-9.238171,False
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,2017-07,16.542245,-5.543113,False
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,2017-04,NaN,NaN,<NA>
7,6514b8ad8028c9f2cc2374ded245783f,delivered,2017-05,9.989826,-11.461215,False


Tasa tardía entre pedidos con duración observada: 8.1%


## 8. Una función reutilizable y exportación trazable
Una función expresa contrato, entradas y salidas. No se necesita programación orientada a objetos para una transformación tabular sencilla. Guardaremos el resultado derivado aparte de los CSV originales, sin sobrescribir las fuentes. El `README` explica cómo reproducirlo.

In [13]:
def resumen_por_estado(df: pd.DataFrame) -> pd.DataFrame:
    """Cuenta pedidos por estado sin confundirlos con clientes únicos."""
    return (df.groupby("customer_state",dropna=False)
              .agg(pedidos=("order_id","nunique"),
                   compradores=("customer_unique_id","nunique"),
                   mediana_valor=("subtotal_productos","median"))
              .sort_values("pedidos",ascending=False))
display(resumen_por_estado(analitica).head(8).round(2))
from json import dumps
outdir = ROOT / "data" / "processed"; outdir.mkdir(parents=True,exist_ok=True)
analitica.to_csv(outdir / "olist_pedidos_analitica.csv",index=False)
(outdir / "origen_tabla_analitica.txt").write_text(
    f"Origen: {ORIGEN}\nTransformación: olist_tools.build_order_level\nUnidad: pedido\n",encoding="utf8")
print("Guardado en data/processed/ (las fuentes originales no se modificaron).")

,pedidos,compradores,mediana_valor
customer_state,,,
SP,41746,40302,79.80
RJ,12852,12384,89.99
MG,11635,11259,89.90
RS,5466,5277,89.00
PR,5045,4882,84.99
SC,3637,3534,89.90
BA,3380,3277,94.90
DF,2140,2075,87.99


Guardado en data/processed/ (las fuentes originales no se modificaron).


### Interpretación con los datos suministrados

En los archivos recibidos aparecen **99 441 pedidos**, **112 650 líneas de pedido** y **103 886 registros de pago**. Esta diferencia no es un error: las tablas tienen **distintas unidades de observación**. La tabla de clientes registra 99 441 identificadores `customer_id`, pero 96 096 identificadores `customer_unique_id` distintos; por ello no debemos llamar «personas» al conteo de registros de clientes.

**Pregunta de reflexión:** ¿por qué una fusión entre pedidos, ítems y pagos, sin agregaciones previas, no permite sumar de manera fiable los importes de compra?


## Interpretación y límites
Una unión correcta no garantiza conclusiones correctas: revisa la población analizada, la cobertura temporal, el contexto y el hecho de que datos de clientes, pedidos, pagos e ítems tienen granularidades distintas. Los resultados numéricos corresponden a los CSV suministrados; las conclusiones son descriptivas, no representativas de todo el comercio electrónico brasileño.

## Ejercicios (para resolver fuera de clase)
- **Esencial.** Filtra los pedidos `delivered` y calcula el conteo por mes. Contrasta dos métodos (`value_counts` y `groupby`).
- **Interpretación.** Explica por qué no se debe sumar `payment_value` después de unir directamente una tabla de pagos a una de ítems.
- **Extensión computacional.** Construye una función que reciba un código de estado y devuelva pedidos, compradores distintos y valor mediano. Agrega una aserción para validar que el filtro corresponde a un estado conocido.

**Referencias:** notas de las presentaciones anteriores (naturaleza, tipos, calidad, implementación práctica); programa docente aprobado; documentación de [pandas](https://pandas.pydata.org/docs/user_guide/index.html) y [Olist/Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).

In [14]:
# Espacio de trabajo (escriba su solución sin modificar las celdas previas):
# 1) Su agrupación por mes aquí.
# 2) Su argumento sobre el doble conteo en Markdown.
# 3) def resumen_estado(df, estado): ...